In [0]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import xml.etree.ElementTree as ET
import urllib3
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    NullType, StringType, DateType,
    DecimalType, IntegerType, LongType, TimestampType
)
from pyspark.sql.functions import col, to_date, current_timestamp, lit, expr

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Config ─────────────────────────────────────────────────────────────────
URL      = "https://poc-sapdevene.enesis.com:8020/sap/opu/odata/sap/ZCDS_VBRP_ODATA_CDS/ZCDS_VBRP_ODATA"
USERNAME = dbutils.secrets.get(scope="sap-odata-username", key="client_secret")
PASSWORD = dbutils.secrets.get(scope="sap-odata-password", key="client_secret")
HEADERS  = {"Accept": "application/atom+xml", "sap-client": "120"}
NS       = {
    "atom": "http://www.w3.org/2005/Atom",
    "m"   : "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata",
    "d"   : "http://schemas.microsoft.com/ado/2007/08/dataservices"
}

vbrp_schema = {
    "Vbeln"                          : StringType(),
    "Posnr"                          : StringType(),        # NUMC
    "Uepos"                          : StringType(),        # NUMC
    "Fkimg"                          : DecimalType(15, 3),  # QUAN — Billing Quantity
    "Vrkme"                          : StringType(),        # UNIT
    "Umvkz"                          : StringType(),        # DEC — keep string
    "Umvkn"                          : StringType(),        # DEC — keep string
    "Meins"                          : StringType(),        # UNIT
    "Smeng"                          : DecimalType(15, 3),  # QUAN
    "Fklmg"                          : DecimalType(15, 3),  # QUAN
    "Lmeng"                          : DecimalType(15, 3),  # QUAN
    "Ntgew"                          : DecimalType(15, 3),  # QUAN — Net Weight
    "Brgew"                          : DecimalType(15, 3),  # QUAN — Gross Weight
    "Gewei"                          : StringType(),        # UNIT
    "Volum"                          : DecimalType(15, 3),  # QUAN — Volume
    "Voleh"                          : StringType(),        # UNIT
    "Gsber"                          : StringType(),
    "Prsdt"                          : DateType(),          # DATS
    "Fbuda"                          : DateType(),          # DATS
    "Kursk"                          : StringType(),        # DEC — keep string
    "Netwr"                          : DecimalType(23, 2),  # CURR
    "Vbelv"                          : StringType(),
    "Posnv"                          : StringType(),        # NUMC
    "Vgbel"                          : StringType(),
    "Vgpos"                          : StringType(),        # NUMC
    "Vgtyp"                          : StringType(),
    "Aubel"                          : StringType(),
    "Aupos"                          : StringType(),        # NUMC
    "Auref"                          : StringType(),
    "Matnr"                          : StringType(),
    "Arktx"                          : StringType(),
    "Pmatn"                          : StringType(),
    "Charg"                          : StringType(),
    "Matkl"                          : StringType(),
    "Pstyv"                          : StringType(),
    "Posar"                          : StringType(),
    "Prodh"                          : StringType(),
    "Vstel"                          : StringType(),
    "Atpkz"                          : StringType(),
    "Spart"                          : StringType(),
    "Pospa"                          : StringType(),        # NUMC
    "Werks"                          : StringType(),
    "Aland"                          : StringType(),
    "Wkreg"                          : StringType(),
    "Wkcou"                          : StringType(),
    "Wkcty"                          : StringType(),
    "Taxm1"                          : StringType(),
    "Taxm2"                          : StringType(),
    "Taxm3"                          : StringType(),
    "Taxm4"                          : StringType(),
    "Taxm5"                          : StringType(),
    "Taxm6"                          : StringType(),
    "Taxm7"                          : StringType(),
    "Taxm8"                          : StringType(),
    "Taxm9"                          : StringType(),
    "Kowrr"                          : StringType(),
    "Prsfd"                          : StringType(),
    "Sktof"                          : StringType(),
    "Skfbp"                          : DecimalType(23, 2),  # CURR
    "Kondm"                          : StringType(),
    "Ktgrm"                          : StringType(),
    "Kostl"                          : StringType(),
    "Bonus"                          : StringType(),
    "Provg"                          : StringType(),
    "Eannr"                          : StringType(),
    "Vkgrp"                          : StringType(),
    "Vkbur"                          : StringType(),
    "Spara"                          : StringType(),
    "Shkzg"                          : StringType(),
    "Ernam"                          : StringType(),
    "Erdat"                          : DateType(),          # DATS — Created On
    "Erzet"                          : StringType(),        # TIMS
    "Bwtar"                          : StringType(),
    "Lgort"                          : StringType(),
    "Stafo"                          : StringType(),
    "Wavwr"                          : DecimalType(23, 2),  # CURR
    "Kzwi1"                          : DecimalType(23, 2),  # CURR
    "Kzwi2"                          : DecimalType(23, 2),  # CURR
    "Kzwi3"                          : DecimalType(23, 2),  # CURR
    "Kzwi4"                          : DecimalType(23, 2),  # CURR
    "Kzwi5"                          : DecimalType(23, 2),  # CURR
    "Kzwi6"                          : DecimalType(23, 2),  # CURR
    "Stcur"                          : StringType(),        # DEC — keep string
    "Uvprs"                          : StringType(),
    "Uvall"                          : StringType(),
    "Ean11"                          : StringType(),
    "Prctr"                          : StringType(),
    "Kvgr1"                          : StringType(),
    "Kvgr2"                          : StringType(),
    "Kvgr3"                          : StringType(),
    "Kvgr4"                          : StringType(),
    "Kvgr5"                          : StringType(),
    "Mvgr1"                          : StringType(),
    "Mvgr2"                          : StringType(),
    "Mvgr3"                          : StringType(),
    "Mvgr4"                          : StringType(),
    "Mvgr5"                          : StringType(),
    "Matwa"                          : StringType(),
    "Bonba"                          : DecimalType(23, 2),  # CURR
    "Kokrs"                          : StringType(),
    "Paobjnr"                        : StringType(),
    "PsPspPnr"                       : StringType(),        # NUMC
    "Aufnr"                          : StringType(),
    "Txjcd"                          : StringType(),
    "Cmpre"                          : DecimalType(23, 2),  # CURR
    "Cmpnt"                          : StringType(),
    "Cuobj"                          : StringType(),        # NUMC
    "CuobjCh"                        : StringType(),        # NUMC
    "Koupd"                          : StringType(),
    "Uecha"                          : StringType(),        # NUMC
    "Xchar"                          : StringType(),
    "Abrvw"                          : StringType(),
    "Sernr"                          : StringType(),
    "BzirkAuft"                      : StringType(),
    "KdgrpAuft"                      : StringType(),
    "KondaAuft"                      : StringType(),
    "LlandAuft"                      : StringType(),
    "Mprok"                          : StringType(),
    "PltypAuft"                      : StringType(),
    "RegioAuft"                      : StringType(),
    "VkorgAuft"                      : StringType(),
    "VtwegAuft"                      : StringType(),
    "Abrbg"                          : DateType(),          # DATS
    "Prosa"                          : StringType(),
    "Uepvw"                          : StringType(),
    "Autyp"                          : StringType(),
    "Stadat"                         : DateType(),          # DATS
    "Fplnr"                          : StringType(),
    "Fpltr"                          : StringType(),        # NUMC
    "Aktnr"                          : StringType(),
    "KnumaPi"                        : StringType(),
    "KnumaAg"                        : StringType(),
    "Mwsbp"                          : DecimalType(23, 2),  # CURR
    "AugruAuft"                      : StringType(),
    "Fareg"                          : StringType(),
    "Upmat"                          : StringType(),
    "Ukonm"                          : StringType(),
    "CmpreFlt"                       : DecimalType(15, 6),  # FLTP
    "Abfor"                          : StringType(),
    "Abges"                          : DecimalType(15, 6),  # FLTP
    "J1arfz"                         : StringType(),
    "J1aregio"                       : StringType(),
    "J1agicd"                        : StringType(),
    "J1adtyp"                        : StringType(),
    "J1atxrel"                       : StringType(),
    "J1bcfop"                        : StringType(),
    "J1btaxlw1"                      : StringType(),
    "J1btaxlw2"                      : StringType(),
    "J1btxsdc"                       : StringType(),
    "Brtwr"                          : DecimalType(23, 2),  # CURR
    "Wktnr"                          : StringType(),
    "Wktps"                          : StringType(),        # NUMC
    "Rplnr"                          : StringType(),
    "KurskDat"                       : DateType(),          # DATS
    "Wgru1"                          : StringType(),
    "Wgru2"                          : StringType(),
    "Kdkg1"                          : StringType(),
    "Kdkg2"                          : StringType(),
    "Kdkg3"                          : StringType(),
    "Kdkg4"                          : StringType(),
    "Kdkg5"                          : StringType(),
    "Vkaus"                          : StringType(),
    "J1aindxp"                       : StringType(),
    "J1aidatep"                      : DateType(),          # DATS
    "Kzfme"                          : StringType(),
    "Mwskz"                          : StringType(),
    "Vertt"                          : StringType(),
    "Vertn"                          : StringType(),
    "Sgtxt"                          : StringType(),
    "Delco"                          : StringType(),
    "Bemot"                          : StringType(),
    "Rrrel"                          : StringType(),
    "Wminr"                          : StringType(),
    "VgbelEx"                        : StringType(),
    "VgposEx"                        : StringType(),        # NUMC
    "Logsys"                         : StringType(),
    "VgtypEx"                        : StringType(),
    "J1btaxlw3"                      : StringType(),
    "J1btaxlw4"                      : StringType(),
    "J1btaxlw5"                      : StringType(),
    "MsrId"                          : StringType(),
    "MsrRefundCode"                  : StringType(),
    "MsrRetReason"                   : StringType(),
    "NrabKnumh"                      : StringType(),
    "NrabValue"                      : DecimalType(23, 2),  # CURR
    "DisputeCase"                    : StringType(),        # RAW — keep string
    "FundUsageItem"                  : StringType(),        # RAW — keep string
    "FarrReltype"                    : StringType(),
    "ClaimsTaxation"                 : StringType(),
    "KurrfDatOrig"                   : DateType(),          # DATS
    "SgtRcat"                        : StringType(),
    "SgtScat"                        : StringType(),
    "Prefe"                          : StringType(),
    "Akkur"                          : StringType(),        # DEC — keep string
    "Waerk"                          : StringType(),        # CUKY
    "Draft"                          : StringType(),
    "Activedocument"                 : StringType(),
    "Grwrt"                          : DecimalType(23, 2),  # CURR
    "Fksaa"                          : StringType(),
    "Absta"                          : StringType(),
    "Abgru"                          : StringType(),
    "Mwsk1"                          : StringType(),
    "Dataaging"                      : DateType(),          # DATS
    "DummyBillgdocitemInclEewPs"     : StringType(),
    "xcwmxmenge"                     : DecimalType(15, 3),  # QUAN
    "xcwmxmeins"                     : StringType(),        # UNIT
    "GloLogRef1It"                   : StringType(),
    "Zapcgki"                        : StringType(),        # NUMC
    "ApcgkExtendi"                   : StringType(),        # NUMC
    "Zabdati"                        : DateType(),          # DATS
    "Aufpl"                          : StringType(),        # NUMC
    "Aplzl"                          : StringType(),        # NUMC
    "Dpcnr"                          : StringType(),
    "Dcpnr"                          : StringType(),        # NUMC
    "Dpnrb"                          : StringType(),        # NUMC
    "Bosfar"                         : StringType(),
    "DpBelnr"                        : StringType(),
    "DpBukrs"                        : StringType(),
    "DpGjahr"                        : StringType(),        # NUMC
    "DpBuzei"                        : StringType(),        # NUMC
    "Packno"                         : StringType(),        # NUMC
    "PeropBeg"                       : DateType(),          # DATS
    "PeropEnd"                       : DateType(),          # DATS
    "FmfgusKey"                      : StringType(),
    "FshSeasonYear"                  : StringType(),
    "FshSeason"                      : StringType(),
    "FshCollection"                  : StringType(),
    "FshTheme"                       : StringType(),
    "Fonds"                          : StringType(),
    "Fistl"                          : StringType(),
    "Fkber"                          : StringType(),
    "GrantNbr"                       : StringType(),
    "BudgetPd"                       : StringType(),
    "J3gbelnri"                      : StringType(),
    "J3gpmaufe"                      : StringType(),
    "J3gpmaufv"                      : StringType(),
    "J3getypa"                       : StringType(),
    "J3getype"                       : StringType(),
    "J3gorgueb"                      : StringType(),
    "PrsWorkPeriod"                  : StringType(),        # NUMC
    "Pprctr"                         : StringType(),
    "Pargb"                          : StringType(),
    "AufplOaa"                       : StringType(),        # NUMC
    "AplzlOaa"                       : StringType(),        # NUMC
    "Campaign"                       : StringType(),        # RAW — keep string
    "Compreas"                       : StringType(),
    "WrfCharstc1"                    : StringType(),
    "WrfCharstc2"                    : StringType(),
    "WrfCharstc3"                    : StringType(),
}
spark = SparkSession.builder.getOrCreate()


# ── Main function ──────────────────────────────────────────────────────────
def ingest_vbrp_bronze(start_date: str, end_date: str, write_mode: str = "overwrite"):
    """
    Fetch VBRP dari SAP OData dan simpan ke Bronze Delta table.

    Parameters
    ----------
    start_date : str  — format 'YYYY-MM-DD', contoh '2019-09-01'
    end_date   : str  — format 'YYYY-MM-DD', contoh '2019-09-30'
    write_mode : str  — 'overwrite' atau 'append' (default: 'overwrite')
    """

    print(f"\n{'='*60}")
    print(f"📦 Fetching VBRP: {start_date} → {end_date}")
    print(f"{'='*60}")

    # ── Fetch dengan pagination ────────────────────────────────
    records = []
    page    = 0

    while True:
        params = {
            "$filter": f"Erdat ge datetime'{start_date}T00:00:00' and Erdat le datetime'{end_date}T23:59:59'",
        }

        response = requests.get(
            URL,
            auth    = HTTPBasicAuth(USERNAME, PASSWORD),
            headers = HEADERS,
            params  = params,
            verify  = False,
            timeout = 300  # naikkan timeout karena data bisa besar
        )
        print(f"  status={response.status_code} | size={len(response.content)} bytes")

        try:
            root = ET.fromstring(response.content)
        except ET.ParseError as e:
            raise RuntimeError(f"Failed to parse XML: {e}")

        root_tag = root.tag.split("}")[-1]
        batch    = []

        if root_tag == "feed":
            for entry in root.findall("atom:entry", NS):
                props = entry.find(".//m:properties", NS)
                if props is not None:
                    batch.append({child.tag.split("}")[-1]: child.text for child in props})
        elif root_tag == "entry":
            props = root.find(".//m:properties", NS)
            if props is not None:
                batch.append({child.tag.split("}")[-1]: child.text for child in props})

        records.extend(batch)
        print(f"  → Fetched: {len(batch):,} rows | Total: {len(records):,}")
        break  # SAP return semua sekaligus, tidak perlu loop
            

    # ── pandas → Spark ─────────────────────────────────────────
    df       = pd.DataFrame(records)
    spark_df = spark.createDataFrame(df)

    # ── Cast schema ────────────────────────────────────────────
    for col_name, col_type in vbrp_schema.items():
        if col_name in spark_df.columns:
            if isinstance(col_type, DateType):
                spark_df = spark_df.withColumn(
                    col_name,
                    expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd\\'T\\'HH:mm:ss')")
                )
            else:
                spark_df = spark_df.withColumn(col_name, col(col_name).cast(col_type))

    # ── Fix NullType ───────────────────────────────────────────
    for field in spark_df.schema.fields:
        if isinstance(field.dataType, NullType):
            spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(StringType()))

    # ── Audit columns ──────────────────────────────────────────
    spark_df = spark_df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source", lit("ZCDS_VBRP_ODATA"))

    # ── Write ──────────────────────────────────────────────────
    spark_df.write \
        .format("delta") \
        .mode(write_mode) \
        .option("overwriteSchema", "true") \
        .saveAsTable("poc_enesis.bronze.ZCDS_VBRP_ODATA")

    print(f"  ✅ Written to Bronze ({write_mode})")
    spark.sql("SELECT COUNT(*) as total FROM poc_enesis.bronze.ZCDS_VBRP_ODATA").show()

In [0]:
print("========= BATCH 1: Januari - September (part 1) =========")
ingest_vbrp_bronze("2019-01-01", "2019-09-14", write_mode="overwrite") 

In [0]:
print("========= BATCH 2: September (part 2)=========")
ingest_vbrp_bronze("2019-09-15", "2019-09-30", write_mode="append") 

In [0]:
print("========= BATCH 3: Oktober (part 1) =========")
ingest_vbrp_bronze("2019-10-01", "2019-10-14", write_mode="append") 

In [0]:
print("========= BATCH 4: Oktober (part 2) =========")
ingest_vbrp_bronze("2019-10-15", "2019-10-31", write_mode="append") 

In [0]:
print("========= BATCH 5: November (part 1) =========")
ingest_vbrp_bronze("2019-11-01", "2019-11-14", write_mode="append") 

In [0]:
print("========= BATCH 6: November (part 2) =========")
ingest_vbrp_bronze("2019-11-15", "2019-11-30", write_mode="append") 

In [0]:
print("========= BATCH 7: Desember (part 1) =========")
ingest_vbrp_bronze("2019-12-01", "2019-12-14", write_mode="append") 

In [0]:
print("========= BATCH 8: Desember (part 2) =========")
ingest_vbrp_bronze("2019-12-15", "2019-12-31", write_mode="append") 

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, to_date

watermark_row = spark.createDataFrame([Row(
    source_name   = "SAP_ODATA",
    table_name    = "ZCDS_VBRP_ODATA",
    last_run_date = RUN_DATE[:10],
    last_run_ts   = None,
    status        = "SUCCESS",
    rows_ingested = int(spark_df.count())
)])

watermark_row = watermark_row \
    .withColumn("last_run_date", to_date("last_run_date")) \
    .withColumn("last_run_ts", current_timestamp())

from delta.tables import DeltaTable
wm = DeltaTable.forName(spark, "poc_enesis.bronze.pipeline_watermark")
wm.alias('t').merge(
    watermark_row.alias('s'),
    't.source_name = s.source_name AND t.table_name = s.table_name'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("✅ Watermark updated")


In [0]:
import uuid
from datetime import datetime

run_id    = str(uuid.uuid4())[:8]
started   = datetime.utcnow()
SCHEMA     = 'BRONZE'
TABLE     = 'ZCDS_VBRP_ODATA'

try:
    # ... semua logic ingest & write ke bronze ...
    rows_in = spark_df.count()
    status  = 'SUCCESS'
    msg     = f'{rows_in} rows ingested'
except Exception as e:
    rows_in = 0
    status  = 'FAILED'
    msg     = str(e)[:500]
    raise  # re-raise supaya Job tau pipeline gagal
finally:
    ended = datetime.utcnow()
    log_df = spark.createDataFrame([{
        'run_id': run_id, 'schema_name': SCHEMA, 'table_name': TABLE,
        'status': status, 'message': msg, 'rows_in': rows_in, 'rows_out': 0,
        'started_at': started, 'ended_at': ended
    }])
    log_df.write.format('delta').mode('append').saveAsTable(
        'poc_enesis.bronze.run_log'
    )
